# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: The Anatomy of Growing Content

The paper reports that growing pages are generally longer, younger, and have slightly better search positions than declining pages.

**Methodology Question**

Where does the label ("growing" vs "declining") come from?

The paper explains that the trend direction is calculated from the change in impressions between the last 30 days and the previous 30 days. I would also like to know whether pages from the same client appeared in both training and validation because this could affect how well the findings generalize.

---

### Finding 2: The Content Performance Curve

The paper finds that content performs best around 61–90 days after publication and generally declines after 270 days, while refreshed older pages may recover.

**Methodology Question**

How was this observation validated?

The finding appears to be based on portfolio-level comparisons rather than a controlled experiment. I would ask whether the comparison controls for factors such as topic, search demand, or competition before concluding that content age alone explains performance changes.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [10]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)

(30000, 44)


In [11]:
df["target"] = (df["trend_direction"] == "down").astype(int)

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[features].fillna(df[features].median(numeric_only=True))
y = df["target"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (30000, 28)
Target: (30000,)


In [12]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("Unique training clients:",
      df.iloc[train_idx]["client_id"].nunique())

print("Unique testing clients:",
      df.iloc[test_idx]["client_id"].nunique())

Training samples: 23837
Testing samples : 6163
Unique training clients: 25
Unique testing clients: 7


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Logistic Regression
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=3000,
        random_state=42
    ))
])

lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

print("Grouped Split - Logistic Regression")
print("-----------------------------------")
print("Accuracy :", accuracy_score(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred))
print("Recall   :", recall_score(y_test, lr_pred))
print("F1 Score :", f1_score(y_test, lr_pred))

print()

# Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("Grouped Split - Random Forest")
print("-----------------------------")
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall   :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))

Grouped Split - Logistic Regression
-----------------------------------
Accuracy : 0.7569365568716534
Precision: 0.8090602770497941
Recall   : 0.6862496030485868
F1 Score : 0.7426116838487973

Grouped Split - Random Forest
-----------------------------
Accuracy : 0.8860944345286387
Precision: 0.8765774084333641
Recall   : 0.904414099714195
F1 Score : 0.8902782119412317


In [14]:
import pandas as pd

comparison = pd.DataFrame({
    "Validation": [
        "Week 5 - Random Split",
        "Week 6 - Grouped Split"
    ],
    "Model": [
        "Random Forest",
        "Random Forest"
    ],
    "Accuracy": [
        0.9098,
        0.8861
    ],
    "Precision": [
        0.8981,
        0.8766
    ],
    "Recall": [
        0.9403,
        0.9044
    ],
    "F1 Score": [
        0.9187,
        0.8903
    ]
})

comparison

,Validation,Model,Accuracy,Precision,Recall,F1 Score
0,Week 5 - Random Split,Random Forest,0.9098,0.8981,0.9403,0.9187
1,Week 6 - Grouped Split,Random Forest,0.8861,0.8766,0.9044,0.8903


In [15]:
print("""
Before vs After Validation

Week 5 used a random train-test split.

Week 6 used a grouped split based on client_id, ensuring that pages from the same client did not appear in both the training and testing sets.

The grouped split produced lower evaluation metrics, indicating a more realistic estimate of model performance on unseen clients.

This validation design better reflects real-world deployment and reduces the risk of memorizing client-specific patterns.
""")


Before vs After Validation

Week 5 used a random train-test split.

Week 6 used a grouped split based on client_id, ensuring that pages from the same client did not appear in both the training and testing sets.

The grouped split produced lower evaluation metrics, indicating a more realistic estimate of model performance on unseen clients.

This validation design better reflects real-world deployment and reduces the risk of memorizing client-specific patterns.



## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [16]:
print("""
Leakage Audit

The final model was reviewed for potential data leakage.

Checks performed:

1. trend_direction was used only as the target label and never as a feature.

2. trend_pct was excluded because it directly contributes to the label definition.

3. No product-generated flags or decision outputs were included as model inputs.

4. Only historical search, traffic and engagement metrics available before the prediction point were used.

5. A grouped validation split by client_id was used to reduce memorization of client-specific patterns.

Conclusion:

No obvious label-derived, future-window or decision-derived leakage was identified in the final feature set.
""")


Leakage Audit

The final model was reviewed for potential data leakage.

Checks performed:

1. trend_direction was used only as the target label and never as a feature.

2. trend_pct was excluded because it directly contributes to the label definition.

3. No product-generated flags or decision outputs were included as model inputs.

4. Only historical search, traffic and engagement metrics available before the prediction point were used.

5. A grouped validation split by client_id was used to reduce memorization of client-specific patterns.

Conclusion:

No obvious label-derived, future-window or decision-derived leakage was identified in the final feature set.



## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [17]:
print("""
Claim Rewrite

Original Claim

'Random Forest is the best model for predicting declining content.'

Rewritten Claim

'In this dataset, Random Forest achieved the highest measured performance among the evaluated models under both random and grouped validation splits.

These results suggest that Random Forest is a useful decision-support model for identifying potentially declining content, although its performance may vary on other datasets or future content.'
""")


Claim Rewrite

Original Claim

'Random Forest is the best model for predicting declining content.'

Rewritten Claim

'In this dataset, Random Forest achieved the highest measured performance among the evaluated models under both random and grouped validation splits.

These results suggest that Random Forest is a useful decision-support model for identifying potentially declining content, although its performance may vary on other datasets or future content.'



## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.